# GNSS Quality Analysis — Google 35-Metric Framework

**Device:** Xiaomi 2201116PI (Xiaomi 13) | Android 13 | GnssLogger v3.1.1.2  
**Chipset:** Qualcomm MPSS.HI.4.3.1  
**Log:** 2026-02-25 17:55 UTC | Bangalore, India  

This notebook evaluates GNSS quality against Google's framework of quality thresholds,  
covering **35 metrics** across 7 categories:

| # | Category | Metrics |
|---|----------|---------|
| A | Clock Quality | 5 |
| B | Tracking State | 5 |
| C | Signal Quality (CN0 / AGC) | 6 |
| D | Pseudorange Rate (Doppler) | 4 |
| E | Carrier Phase / ADR | 4 |
| F | Feature & Frequency Support | 8 |
| G | Fix Quality | 3 |
| | **Total** | **35** |

---
## Setup & Data Loading

In [ ]:
import os, warnings, statistics
warnings.filterwarnings('ignore')
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pynmea2

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

LOG_DIR    = os.path.join(os.getcwd(), 'Log1')
TXT_FILE   = os.path.join(LOG_DIR, 'gnss_log_2026_02_25_23_25_37.txt')
NMEA_FILE  = os.path.join(LOG_DIR, 'gnss_log_2026_02_25_23_25_37.nmea')
RINEX_FILE = os.path.join(LOG_DIR, 'gnss_log_2026_02_25_23_25_37.26o')

CMAP_INT  = {1:'GPS', 3:'GLONASS', 4:'QZSS', 5:'BeiDou', 6:'Galileo', 7:'NavIC'}
CCOLOUR   = {'GPS':'#1f77b4','GLONASS':'#ff7f0e','BeiDou':'#d62728',
             'Galileo':'#2ca02c','QZSS':'#9467bd'}

# ── Parse Raw rows ────────────────────────────────────────────────────────────
raw_header = None
raw_rows   = []
with open(TXT_FILE) as f:
    for line in f:
        if line.startswith('# Raw,'):
            raw_header = line.strip()[2:].split(',')
        elif line.startswith('Raw,') and raw_header:
            v = line.strip().split(',')
            if len(v) == len(raw_header):
                raw_rows.append(dict(zip(raw_header, v)))

def fv(row, key):
    try: return float(row.get(key,'') or 'x')
    except: return None
def iv(row, key, default=0):
    try: return int(row.get(key,'') or 'x')
    except: return default

print(f'Raw rows parsed  : {len(raw_rows)}')
print(f'Raw fields       : {len(raw_header)}')
ct = Counter(CMAP_INT.get(iv(r,'ConstellationType'),'?') for r in raw_rows)
for c,n in sorted(ct.items()): print(f'  {c:10s}: {n}')

---
## Category A — Clock Quality

> **Metrics A1–A5** | Evaluates the stability and reliability of the receiver's hardware clock.

In [ ]:
# ── A1: BiasUncertaintyNanos ──────────────────────────────────────────────────
bias_unc = [fv(r,'BiasUncertaintyNanos') for r in raw_rows if fv(r,'BiasUncertaintyNanos') is not None]
a1_mean  = statistics.mean(bias_unc)
a1_max   = max(bias_unc)
a1_pass  = a1_max < 40

# ── A2: HardwareClockDiscontinuityCount (new jumps in session) ────────────────
disc_vals = [iv(r,'HardwareClockDiscontinuityCount') for r in raw_rows]
a2_session_jumps = max(disc_vals) - min(disc_vals)
a2_pass = a2_session_jumps == 0

# ── A3: LeapSecond reported ───────────────────────────────────────────────────
ls = [iv(r,'LeapSecond') for r in raw_rows if r.get('LeapSecond','') not in ('','0')]
a3_pass = len(ls) > 0

# ── A4: FullBias + Bias stability (no internal clock jumps) ──────────────────
total_bias = [(fv(r,'FullBiasNanos') or 0) + (fv(r,'BiasNanos') or 0) for r in raw_rows]
bias_diffs = [abs(total_bias[i]-total_bias[i-1]) for i in range(1,len(total_bias))]
a4_max_jump_ns = max(bias_diffs)
a4_pass = a4_max_jump_ns < 1e9   # < 1 second jump

# ── A5: DriftUncertaintyNanosPerSecond reported ───────────────────────────────
drift_unc = [fv(r,'DriftUncertaintyNanosPerSecond') for r in raw_rows if fv(r,'DriftUncertaintyNanosPerSecond') is not None]
drift_vals= [fv(r,'DriftNanosPerSecond') for r in raw_rows if fv(r,'DriftNanosPerSecond') is not None]
a5_pass   = len(drift_unc) == len(raw_rows)

print('='*62)
print('  CATEGORY A — CLOCK QUALITY')
print('='*62)
def pf(p): return '  PASS  ' if p else '  FAIL  '
print(f'A1 BiasUncertaintyNanos < 40 ns     : {pf(a1_pass)}  mean={a1_mean:.1f} max={a1_max:.1f} ns')
print(f'A2 No new clock discontinuities     : {pf(a2_pass)}  session jumps={a2_session_jumps}  (counter at boot={min(disc_vals)})')
print(f'A3 LeapSecond reported              : {pf(a3_pass)}  value={sorted(set(ls))}')
print(f'A4 Bias stability (no jumps > 1 s)  : {pf(a4_pass)}  max jump={a4_max_jump_ns:.0f} ns')
print(f'A5 DriftUncertainty reported        : {pf(a5_pass)}  mean={statistics.mean(drift_unc):.2f} ns/s')

# ── Visualisation: Bias uncertainty over time ─────────────────────────────────
t_ms  = np.array([fv(r,'utcTimeMillis') for r in raw_rows])
t_sec = (t_ms - t_ms[0]) / 1000.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.scatter(t_sec, bias_unc, s=8, alpha=0.8, color='#1f77b4')
ax.axhline(40,  color='red',    linewidth=1.2, linestyle='--', label='40 ns threshold (FAIL)')
ax.axhline(150, color='orange', linewidth=1.0, linestyle=':', label='150 ns relaxed threshold')
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('BiasUncertaintyNanos (ns)')
ax.set_title('A1 — Clock Bias Uncertainty over Time')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[1]
drift_arr = np.array(drift_vals)
ax.plot(t_sec[:len(drift_vals)], drift_arr, color='#2ca02c', linewidth=1.2)
ax.axhline(0, color='red', linewidth=0.8, linestyle='--')
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('Clock Drift (ns/s)')
ax.set_title('A5 — Clock Frequency Drift over Time')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Category B — Tracking State Completeness

> **Metrics B1–B5** | Checks that measurements have complete and valid tracking state flags.

In [ ]:
STATE_CODE_LOCK   = 1
STATE_TOW_DECODED = 8
STATE_TOW_KNOWN   = 16384
STATE_GLO_TOD     = 128

states = np.array([iv(r,'State') for r in raw_rows])
n      = len(states)
ctype  = np.array([iv(r,'ConstellationType') for r in raw_rows])

# ── B1: STATE_TOW_DECODED for GPS ────────────────────────────────────────────
gps_mask     = ctype == 1
gps_tow      = np.sum((states[gps_mask] & STATE_TOW_DECODED) > 0)
b1_pct       = 100*gps_tow/np.sum(gps_mask)
b1_pass      = b1_pct >= 99

# ── B2: STATE_GLO_TOD_DECODED for GLONASS ────────────────────────────────────
glo_mask     = ctype == 3
glo_tod      = np.sum((states[glo_mask] & STATE_GLO_TOD) > 0)
b2_pct       = 100*glo_tod/np.sum(glo_mask)
b2_pass      = b2_pct >= 99

# ── B3: ReceivedSvTimeUncertaintyNanos < 500 ns ───────────────────────────────
svtu = np.array([fv(r,'ReceivedSvTimeUncertaintyNanos') for r in raw_rows if fv(r,'ReceivedSvTimeUncertaintyNanos') is not None])
b3_pass = float(np.max(svtu)) < 500

# ── B4: IsFullTracking reported ───────────────────────────────────────────────
ft_vals = [r.get('IsFullTracking','').strip() for r in raw_rows]
ft_reported = sum(1 for x in ft_vals if x in ('0','1','true','false','True','False'))
b4_pass = ft_reported > 0

# ── B5: MultipathIndicator useful (not always UNKNOWN) ───────────────────────
mp_vals = [iv(r,'MultipathIndicator') for r in raw_rows]
mp_not_unknown = sum(1 for x in mp_vals if x in (1,2))
b5_pass = mp_not_unknown > 0

print('='*62)
print('  CATEGORY B — TRACKING STATE COMPLETENESS')
print('='*62)
print(f'B1 GPS STATE_TOW_DECODED ≥ 99%      : {pf(b1_pass)}  {b1_pct:.1f}% ({gps_tow}/{int(np.sum(gps_mask))})')
print(f'B2 GLONASS GLO_TOD_DECODED ≥ 99%    : {pf(b2_pass)}  {b2_pct:.1f}% ({glo_tod}/{int(np.sum(glo_mask))})')
print(f'B3 SvTimeUncertainty < 500 ns       : {pf(b3_pass)}  max={np.max(svtu):.0f} mean={np.mean(svtu):.1f} ns')
print(f'B4 IsFullTracking field reported    : {pf(b4_pass)}  {ft_reported}/{n} rows have value')
print(f'B5 MultipathIndicator != UNKNOWN    : {pf(b5_pass)}  {mp_not_unknown}/{n} non-unknown')

# ── State bitmask bar chart ────────────────────────────────────────────────────
state_flags = {
    'CODE_LOCK(1)': STATE_CODE_LOCK,
    'TOW_DECODED(8)': STATE_TOW_DECODED,
    'TOW_KNOWN(16384)': STATE_TOW_KNOWN,
    'GLO_TOD(128)': STATE_GLO_TOD,
}
fig, ax = plt.subplots(figsize=(9, 3.5))
labels, pcts = [], []
for name, flag in state_flags.items():
    cnt = int(np.sum((states & flag) > 0))
    labels.append(name); pcts.append(100*cnt/n)
bars = ax.barh(labels, pcts, color=['#2ca02c' if p >= 99 else '#d62728' for p in pcts], alpha=0.8)
ax.axvline(99, color='black', linewidth=1, linestyle='--', label='99% threshold')
ax.set_xlabel('% of measurements with flag set')
ax.set_title('B — Tracking State Flags Coverage')
ax.set_xlim(0, 110); ax.legend(fontsize=8)
for bar, pct in zip(bars, pcts):
    ax.text(pct+0.5, bar.get_y()+bar.get_height()/2, f'{pct:.0f}%', va='center', fontsize=9)
ax.grid(True, axis='x', alpha=0.3); plt.tight_layout(); plt.show()

---
## Category C — Signal Quality (CN0, AGC, SNR)

> **Metrics C1–C6** | Carrier-to-noise density is the primary signal quality indicator.

In [ ]:
cn0_all = np.array([fv(r,'Cn0DbHz') for r in raw_rows if fv(r,'Cn0DbHz') is not None])
bb_cn0  = np.array([fv(r,'BasebandCn0DbHz') for r in raw_rows if fv(r,'BasebandCn0DbHz') is not None])
agc_all = [fv(r,'AgcDb') for r in raw_rows if fv(r,'AgcDb') is not None]
snr_all = [fv(r,'SnrInDb') for r in raw_rows if fv(r,'SnrInDb') is not None and fv(r,'SnrInDb') != 0]

# Top-5 mean CN0 (best geometry approximation)
cn0_top5_mean = float(np.mean(sorted(cn0_all, reverse=True)[:5]))

# ── C1: Mean CN0 ≥ 20 dB-Hz (minimum usability) ──────────────────────────────
c1_val  = float(np.mean(cn0_all))
c1_pass = c1_val >= 20

# ── C2: CN0 top-5 mean ≥ 25 dB-Hz (open sky threshold) ──────────────────────
c2_pass = cn0_top5_mean >= 25

# ── C3: Fraction CN0 ≥ 25 dB-Hz ≥ 50% ───────────────────────────────────────
c3_pct  = 100*float(np.sum(cn0_all >= 25))/len(cn0_all)
c3_pass = c3_pct >= 50

# ── C4: Fraction CN0 ≥ 20 dB-Hz ≥ 80% ───────────────────────────────────────
c4_pct  = 100*float(np.sum(cn0_all >= 20))/len(cn0_all)
c4_pass = c4_pct >= 80

# ── C5: BasebandCn0DbHz reported ─────────────────────────────────────────────
c5_pass = len(bb_cn0) == len(raw_rows)

# ── C6: AGC (AgcDb) reported ─────────────────────────────────────────────────
agc_nonzero = [x for x in agc_all if x != 0]
c6_pass = len(agc_nonzero) > len(raw_rows) * 0.9

print('='*62)
print('  CATEGORY C — SIGNAL QUALITY')
print('='*62)
print(f'C1 Mean CN0 ≥ 20 dB-Hz              : {pf(c1_pass)}  {c1_val:.1f} dB-Hz')
print(f'C2 Top-5 CN0 mean ≥ 25 dB-Hz        : {pf(c2_pass)}  {cn0_top5_mean:.1f} dB-Hz')
print(f'C3 CN0 ≥ 25 dB-Hz fraction ≥ 50%    : {pf(c3_pass)}  {c3_pct:.1f}%')
print(f'C4 CN0 ≥ 20 dB-Hz fraction ≥ 80%    : {pf(c4_pass)}  {c4_pct:.1f}%')
print(f'C5 BasebandCn0DbHz reported          : {pf(c5_pass)}  {len(bb_cn0)}/{len(raw_rows)} rows')
print(f'C6 AGC (AgcDb) reported ≥ 90%        : {pf(c6_pass)}  {len(agc_nonzero)}/{len(raw_rows)} non-zero')

# ── Plots ────────────────────────────────────────────────────────────────────
ctype_arr = np.array([iv(r,'ConstellationType') for r in raw_rows])
t_arr     = (np.array([fv(r,'utcTimeMillis') for r in raw_rows]) - fv(raw_rows[0],'utcTimeMillis')) / 1000

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# CN0 by constellation over time
ax = axes[0]
for ct_id, name in CMAP_INT.items():
    mask = ctype_arr == ct_id
    if np.sum(mask) > 0:
        ax.scatter(t_arr[mask], cn0_all[mask], s=8, alpha=0.8,
                   color=CCOLOUR.get(name,'#555'), label=name)
ax.axhline(25, color='red', linewidth=1.2, linestyle='--', label='25 dB-Hz threshold')
ax.axhline(20, color='orange', linewidth=1.0, linestyle=':', label='20 dB-Hz min')
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('CN0 (dB-Hz)')
ax.set_title('C1-C3 — CN0 by Constellation'); ax.legend(markerscale=2, fontsize=7)
ax.grid(True, alpha=0.3)

# CN0 vs BasebandCn0
ax = axes[1]
ax.scatter(cn0_all, bb_cn0, s=6, alpha=0.7, color='#9467bd')
lim = [min(cn0_all.min(), bb_cn0.min())-1, max(cn0_all.max(), bb_cn0.max())+1]
ax.plot(lim, lim, 'r--', linewidth=0.8, label='y=x')
ax.set_xlabel('CN0 (dB-Hz)'); ax.set_ylabel('Baseband CN0 (dB-Hz)')
ax.set_title('C5 — CN0 vs Baseband CN0'); ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# AGC distribution
ax = axes[2]
if agc_nonzero:
    ax.hist(agc_nonzero, bins=20, color='#8c564b', alpha=0.8, edgecolor='white')
    ax.set_xlabel('AGC (dB)'); ax.set_ylabel('Count')
    ax.set_title('C6 — AGC Distribution (narrow range = stable RF)')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Category D — Pseudorange Rate (Doppler) Quality

> **Metrics D1–D4** | Evaluates Doppler measurement quality and chipset limitations.

In [ ]:
prr     = np.array([fv(r,'PseudorangeRateMetersPerSecond')          for r in raw_rows if fv(r,'PseudorangeRateMetersPerSecond') is not None])
prr_unc = np.array([fv(r,'PseudorangeRateUncertaintyMetersPerSecond') for r in raw_rows if fv(r,'PseudorangeRateUncertaintyMetersPerSecond') is not None])

# ── D1: PRR available for all measurements ───────────────────────────────────
d1_pass = len(prr) == len(raw_rows)

# ── D2: PRR uncertainty mean ≤ 1 m/s ─────────────────────────────────────────
d2_val  = float(np.mean(prr_unc))
d2_pass = d2_val <= 1.0

# ── D3: PRR not clamped (< 5% of values within 5 m/s of ±500 m/s) ───────────
clamped = np.sum(np.abs(np.abs(prr) - 500) < 5)
d3_clamp_pct = 100*clamped/len(prr)
d3_pass = d3_clamp_pct < 5

# ── D4: PRR uncertainty max ≤ 5 m/s ──────────────────────────────────────────
d4_val  = float(np.max(prr_unc))
d4_pass = d4_val <= 5.0

print('='*62)
print('  CATEGORY D — PSEUDORANGE RATE (DOPPLER) QUALITY')
print('='*62)
print(f'D1 PRR available for all meas        : {pf(d1_pass)}  {len(prr)}/{len(raw_rows)}')
print(f'D2 PRR uncertainty mean ≤ 1 m/s      : {pf(d2_pass)}  {d2_val:.2f} m/s')
print(f'D3 PRR not clamped at ±500 m/s       : {pf(d3_pass)}  {clamped} values clamped ({d3_clamp_pct:.1f}%)')
print(f'D4 PRR uncertainty max ≤ 5 m/s       : {pf(d4_pass)}  max={d4_val:.2f} m/s')

# ── Plots ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
for ct_id, name in CMAP_INT.items():
    mask = ctype_arr == ct_id
    if np.sum(mask): ax.scatter(t_arr[mask], prr[mask], s=6, alpha=0.7,
                                color=CCOLOUR.get(name,'#555'), label=name)
ax.axhline(500, color='red', linewidth=1, linestyle='--', label='±500 m/s clamp')
ax.axhline(-500, color='red', linewidth=1, linestyle='--')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('PRR (m/s)')
ax.set_title('D3 — Pseudorange Rate with Clamp Limits'); ax.legend(markerscale=2, fontsize=7)
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.hist(prr_unc, bins=25, color='#d62728', alpha=0.8, edgecolor='white')
ax.axvline(1.0, color='red', linewidth=1.2, linestyle='--', label='1 m/s threshold')
ax.axvline(5.0, color='orange', linewidth=1.0, linestyle=':', label='5 m/s max threshold')
ax.set_xlabel('PRR Uncertainty (m/s)'); ax.set_ylabel('Count')
ax.set_title('D2/D4 — PRR Uncertainty Distribution'); ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Category E — Carrier Phase / ADR Quality

> **Metrics E1–E4** | Carrier phase enables centimetre-level positioning. Critical for DGNSS/RTK.

In [ ]:
ADR_VALID      = 1
ADR_RESET      = 2
ADR_CYCLE_SLIP = 4

adr_state = np.array([iv(r,'AccumulatedDeltaRangeState') for r in raw_rows])
adr_valid = int(np.sum((adr_state & ADR_VALID) > 0))
adr_slip  = int(np.sum((adr_state & ADR_CYCLE_SLIP) > 0))
adr_reset = int(np.sum((adr_state & ADR_RESET) > 0))
n_total   = len(adr_state)

# ── E1: ADR measurements available ───────────────────────────────────────────
e1_pass = adr_valid > 0

# ── E2: ADR availability rate ≥ 50% ──────────────────────────────────────────
e2_avail_pct = 100*adr_valid/n_total
e2_pass = e2_avail_pct >= 50

# ── E3: Cycle slip rate < 0.5% of valid ADR ──────────────────────────────────
e3_slip_pct = (100*adr_slip/adr_valid) if adr_valid > 0 else None
e3_pass = (e3_slip_pct is not None and e3_slip_pct < 0.5) if adr_valid > 0 else False

# ── E4: ADR uncertainty ≤ 0.5 m when valid ───────────────────────────────────
adr_unc_valid = [fv(r,'AccumulatedDeltaRangeUncertaintyMeters') for r in raw_rows
                 if (iv(r,'AccumulatedDeltaRangeState') & ADR_VALID) and
                    fv(r,'AccumulatedDeltaRangeUncertaintyMeters') is not None]
e4_pass = len(adr_unc_valid) > 0 and max(adr_unc_valid) <= 0.5

print('='*62)
print('  CATEGORY E — CARRIER PHASE / ADR QUALITY')
print('='*62)
print(f'E1 ADR measurements available        : {pf(e1_pass)}  {adr_valid}/{n_total} valid')
print(f'E2 ADR availability ≥ 50%            : {pf(e2_pass)}  {e2_avail_pct:.1f}%')
print(f'E3 Cycle slip rate < 0.5%            : {"  N/A   " if not e1_pass else pf(e3_pass)}  {"-" if e3_slip_pct is None else f"{e3_slip_pct:.2f}%"}')
print(f'E4 ADR uncertainty ≤ 0.5 m           : {"  N/A   " if not e1_pass else pf(e4_pass)}')
print()
print('  NOTE: This device exposes NO carrier phase (ADR) measurements.')
print('  ADR requires explicit chipset support and driver enablement.')
print('  For best DGNSS performance, carrier phase support is required.')

# ── ADR state pie chart ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))
adr_none = n_total - adr_valid
sizes  = [adr_valid, adr_none]
colors = ['#2ca02c', '#d62728']
labels = [f'ADR Valid ({adr_valid})', f'ADR Not Available ({adr_none})']
wedge_props = dict(width=0.4)
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
       startangle=90, wedgeprops=wedge_props)
ax.set_title('E1/E2 — Carrier Phase (ADR) Availability\n(Donut: outer=ADR valid, inner=total)')
plt.tight_layout(); plt.show()

---
## Category F — Feature & Frequency Support

> **Metrics F1–F8** | Binary feature availability checks.

In [ ]:
# ── F1: L1 frequency support ──────────────────────────────────────────────────
freqs = set(round((fv(r,'CarrierFrequencyHz') or 0)/1e6, 2) for r in raw_rows if fv(r,'CarrierFrequencyHz'))
L1_MHz = 1575.42; L5_MHz = 1176.45
f1_pass = any(abs(f-L1_MHz) < 1 for f in freqs)

# ── F2: L5 frequency support ──────────────────────────────────────────────────
f2_pass = any(abs(f-L5_MHz) < 1 for f in freqs)

# ── F3: SatPvt (satellite ECEF position) reported ────────────────────────────
sv_x = [fv(r,'SvPositionEcefXMeters') for r in raw_rows if fv(r,'SvPositionEcefXMeters') is not None]
f3_pass = sum(1 for x in sv_x if x and x != 0) > 0.8 * len(raw_rows)

# ── F4: SatClockBias reported ─────────────────────────────────────────────────
sv_cb = [fv(r,'SvClockBiasMeters') for r in raw_rows if fv(r,'SvClockBiasMeters') is not None]
f4_pass = sum(1 for x in sv_cb if x and x != 0) > 0.8 * len(raw_rows)

# ── F5: InterSignalBias (FullISB) reported ────────────────────────────────────
isb = [fv(r,'FullInterSignalBiasNanos') for r in raw_rows if fv(r,'FullInterSignalBiasNanos') is not None]
isb_nz = sum(1 for x in isb if x and x != 0)
f5_pass = isb_nz > 0.5 * len(raw_rows)

# ── F6: GPS supported ─────────────────────────────────────────────────────────
f6_pass = any(iv(r,'ConstellationType') == 1 for r in raw_rows)

# ── F7: Non-GPS constellation supported (GLONASS/BeiDou/Galileo) ─────────────
f7_pass = any(iv(r,'ConstellationType') in (3,5,6) for r in raw_rows)

# ── F8: Klobuchar ionosphere model reported ───────────────────────────────────
klob = sum(1 for r in raw_rows if r.get('KlobucharAlpha0','') not in ('','0'))
f8_pass = klob == len(raw_rows)

print('='*62)
print('  CATEGORY F — FEATURE & FREQUENCY SUPPORT')
print('='*62)
print(f'F1 L1 frequency (1575.42 MHz)        : {pf(f1_pass)}  freqs={sorted(freqs)}')
print(f'F2 L5 frequency (1176.45 MHz)        : {pf(f2_pass)}  L1-only device')
print(f'F3 SatPvt ECEF position reported     : {pf(f3_pass)}  {sum(1 for x in sv_x if x)}/{len(raw_rows)}')
print(f'F4 SatClockBias reported             : {pf(f4_pass)}  {sum(1 for x in sv_cb if x)}/{len(raw_rows)}')
print(f'F5 InterSignalBias reported          : {pf(f5_pass)}  {isb_nz}/{len(raw_rows)} non-zero')
print(f'F6 GPS constellation supported       : {pf(f6_pass)}')
print(f'F7 Non-GPS constellation supported   : {pf(f7_pass)}')
print(f'F8 Klobuchar iono model reported     : {pf(f8_pass)}  {klob}/{len(raw_rows)}')

# Feature support bar chart
features = ['L1 Support','L5 Support','SatPvt (ECEF)','SatClockBias',
            'InterSignalBias','GPS','Non-GPS Consts','Klobuchar']
passes   = [f1_pass,f2_pass,f3_pass,f4_pass,f5_pass,f6_pass,f7_pass,f8_pass]

fig, ax = plt.subplots(figsize=(9, 3.5))
colors_bar = ['#2ca02c' if p else '#d62728' for p in passes]
bars = ax.barh(features, [1]*len(features), color=colors_bar, alpha=0.8)
for bar, passed in zip(bars, passes):
    ax.text(0.02, bar.get_y()+bar.get_height()/2,
            'PASS' if passed else 'FAIL', va='center', fontsize=10, fontweight='bold',
            color='white')
ax.set_xlim(0, 1.3); ax.set_xticks([])
ax.set_title('F — Feature & Frequency Support Scorecard')
plt.tight_layout(); plt.show()

---
## Category G — Fix Quality

> **Metrics G1–G3** | Position accuracy and fix consistency.

In [ ]:
import pynmea2

# Parse GGA sentences
gga_rows = []
with open(NMEA_FILE) as f:
    for line in f:
        if not line.startswith('NMEA,'): continue
        parts = line.strip().split(',')
        if 'GGA' not in parts[1]: continue
        try:
            msg = pynmea2.parse(','.join(parts[1:-1]))
            if msg.latitude:
                gga_rows.append({'lat':msg.latitude,'lon':msg.longitude,
                                 'alt':float(msg.altitude),'hdop':float(msg.horizontal_dil),
                                 'sats':int(msg.num_sats),'ts':float(parts[-1])})
        except: pass

lats = np.array([r['lat'] for r in gga_rows])
lons = np.array([r['lon'] for r in gga_rows])
hdops= np.array([r['hdop'] for r in gga_rows])
alts = np.array([r['alt'] for r in gga_rows])

# CEP computation (stationary receiver = scatter = noise)
lat_c, lon_c = lats.mean(), lons.mean()
dist2d = np.sqrt(((lats-lat_c)*111111)**2 + ((lons-lon_c)*108131)**2)
cep50 = float(np.percentile(dist2d, 50))
cep68 = float(np.percentile(dist2d, 68))
cep95 = float(np.percentile(dist2d, 95))

# ── G1: CEP68 (horizontal) ≤ 3 m (open sky stationary) ──────────────────────
g1_pass = cep68 <= 3.0

# ── G2: HDOP ≤ 2.0 in ≥ 95% of fixes ────────────────────────────────────────
g2_pct  = 100*float(np.sum(hdops <= 2.0))/len(hdops)
g2_pass = g2_pct >= 95

# ── G3: Multi-constellation fix (≥ 2 systems in NMEA GSA) ────────────────────
gsa_sys = set()
with open(NMEA_FILE) as f:
    for line in f:
        if not line.startswith('NMEA,'): continue
        parts = line.strip().split(',')
        if 'GSA' not in parts[1]: continue
        # GNGSA has system ID as last field before checksum
        m = parts[1]  # e.g. $GNGSA
        raw_nmea = ','.join(parts[1:-1])
        if raw_nmea.count(',') >= 17:
            sys_id_field = raw_nmea.split(',')[-1]
            try: gsa_sys.add(int(sys_id_field.split('*')[0].strip()))
            except: pass
g3_pass = len(gsa_sys) >= 2

print('='*62)
print('  CATEGORY G — FIX QUALITY')
print('='*62)
print(f'G1 CEP68 (stationary, open sky) ≤ 3m: {pf(g1_pass)}  CEP50={cep50:.3f}m CEP68={cep68:.3f}m CEP95={cep95:.3f}m')
print(f'G2 HDOP ≤ 2.0 in ≥ 95% of fixes     : {pf(g2_pass)}  {g2_pct:.0f}% of fixes  mean HDOP={hdops.mean():.2f}')
print(f'G3 Multi-constellation fix           : {pf(g3_pass)}  {len(gsa_sys)} systems in GSA: {sorted(gsa_sys)}')
print(f'   Altitude noise (std)              :          {float(np.std(alts)):.3f} m')

# ── Plots ────────────────────────────────────────────────────────────────────
t_gga = (np.array([r['ts'] for r in gga_rows]) - gga_rows[0]['ts']) / 1000

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
sc = ax.scatter((lons-lon_c)*108131*100, (lats-lat_c)*111111*100,
                c=dist2d, cmap='RdYlGn_r', s=20, alpha=0.85)
fig.colorbar(sc, ax=ax, label='2D error (cm)')
circle68 = plt.Circle((0,0), cep68*100, fill=False, color='blue', linewidth=1.5, label=f'CEP68={cep68*100:.1f}cm')
ax.add_patch(circle68)
ax.set_xlabel('East offset (cm)'); ax.set_ylabel('North offset (cm)')
ax.set_title('G1 — Fix Scatter (stationary = noise floor)')
ax.legend(fontsize=8); ax.set_aspect('equal'); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(t_gga, hdops, color='#d62728', linewidth=1.3)
ax.axhline(1.0, color='green', linewidth=0.8, linestyle='--', label='HDOP=1.0 (ideal)')
ax.axhline(2.0, color='orange', linewidth=0.8, linestyle='--', label='HDOP=2.0 (threshold)')
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('HDOP')
ax.set_title('G2 — HDOP over Time'); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(t_gga, alts, color='#4c72b0', linewidth=1.3)
ax.fill_between(t_gga, alts, alts.min()-0.3, alpha=0.15, color='#4c72b0')
ax.set_xlabel('Elapsed time (s)'); ax.set_ylabel('Altitude MSL (m)')
ax.set_title(f'G1 — Altitude Noise  std={float(np.std(alts)):.3f} m')
ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## Final Quality Scorecard — All 35 Metrics

In [ ]:
# Build scorecard table
scorecard = [
    # ID, Category, Metric, Threshold, Measured, Pass
    ('A1','Clock',   'BiasUncertaintyNanos',            '< 40 ns',        f'{a1_mean:.1f} ns (max {a1_max:.1f})',   a1_pass),
    ('A2','Clock',   'Clock Discontinuities (session)', '= 0 new',        f'{a2_session_jumps} new (counter={min(disc_vals)})', a2_pass),
    ('A3','Clock',   'LeapSecond reported',             'non-zero',       f'{sorted(set(ls))}',                     a3_pass),
    ('A4','Clock',   'Bias stability (no jumps)',        '< 1s jump',      f'max jump = {a4_max_jump_ns:.0f} ns',   a4_pass),
    ('A5','Clock',   'DriftUncertainty reported',        'present',        f'mean = {statistics.mean(drift_unc):.2f} ns/s', a5_pass),
    ('B1','Tracking','GPS TOW_DECODED rate',             '≥ 99%',          f'{b1_pct:.1f}%',                         b1_pass),
    ('B2','Tracking','GLONASS GLO_TOD rate',             '≥ 99%',          f'{b2_pct:.1f}%',                         b2_pass),
    ('B3','Tracking','SvTimeUncertaintyNanos',           '< 500 ns',       f'max={np.max(svtu):.0f} ns',             b3_pass),
    ('B4','Tracking','IsFullTracking reported',          'present',        f'{ft_reported}/{len(raw_rows)} rows',    b4_pass),
    ('B5','Tracking','MultipathIndicator useful',        'not all UNKNOWN',f'{mp_not_unknown}/{len(raw_rows)} known',b5_pass),
    ('C1','Signal',  'Mean CN0 ≥ 20 dB-Hz',             '≥ 20 dB-Hz',     f'{c1_val:.1f} dB-Hz',                   c1_pass),
    ('C2','Signal',  'Top-5 CN0 mean ≥ 25 dB-Hz',       '≥ 25 dB-Hz',     f'{cn0_top5_mean:.1f} dB-Hz',            c2_pass),
    ('C3','Signal',  'CN0 ≥ 25 dB-Hz fraction',         '≥ 50%',          f'{c3_pct:.1f}%',                         c3_pass),
    ('C4','Signal',  'CN0 ≥ 20 dB-Hz fraction',         '≥ 80%',          f'{c4_pct:.1f}%',                         c4_pass),
    ('C5','Signal',  'BasebandCn0 reported',             '= 100%',         f'{len(bb_cn0)}/{len(raw_rows)}',         c5_pass),
    ('C6','Signal',  'AGC (AgcDb) reported',             '≥ 90%',          f'{len(agc_nonzero)}/{len(raw_rows)}',    c6_pass),
    ('D1','PRR',     'PRR available',                    '= 100%',         f'{len(prr)}/{len(raw_rows)}',            d1_pass),
    ('D2','PRR',     'PRR uncertainty mean ≤ 1 m/s',    '≤ 1 m/s',        f'{d2_val:.2f} m/s',                     d2_pass),
    ('D3','PRR',     'PRR not clamped at ±500 m/s',     '< 5% clamped',   f'{d3_clamp_pct:.1f}%',                  d3_pass),
    ('D4','PRR',     'PRR uncertainty max ≤ 5 m/s',     '≤ 5 m/s',        f'{d4_val:.2f} m/s',                     d4_pass),
    ('E1','ADR',     'ADR measurements available',       '> 0',            f'{adr_valid}/{n_total}',                 e1_pass),
    ('E2','ADR',     'ADR availability ≥ 50%',           '≥ 50%',          f'{e2_avail_pct:.1f}%',                  e2_pass),
    ('E3','ADR',     'Cycle slip rate < 0.5%',           '< 0.5%',         'N/A (no ADR)',                           e3_pass),
    ('E4','ADR',     'ADR uncertainty ≤ 0.5 m',         '≤ 0.5 m',        'N/A (no ADR)',                           e4_pass),
    ('F1','Feature', 'L1 frequency support',             'present',        f'{sorted(freqs)}',                       f1_pass),
    ('F2','Feature', 'L5 frequency support',             'present',        'Not supported',                          f2_pass),
    ('F3','Feature', 'SatPvt ECEF position',             '≥ 80%',          f'{sum(1 for x in sv_x if x)}/{len(raw_rows)}', f3_pass),
    ('F4','Feature', 'SatClockBias reported',            '≥ 80%',          f'{sum(1 for x in sv_cb if x)}/{len(raw_rows)}',f4_pass),
    ('F5','Feature', 'InterSignalBias reported',         '≥ 50% non-zero', f'{isb_nz}/{len(raw_rows)}',              f5_pass),
    ('F6','Feature', 'GPS supported',                    'present',        'Present',                                f6_pass),
    ('F7','Feature', 'Non-GPS constellation',            'present',        'GLONASS+BeiDou+Galileo+QZSS',            f7_pass),
    ('F8','Feature', 'Klobuchar iono model',             '= 100%',         f'{klob}/{len(raw_rows)}',                f8_pass),
    ('G1','Fix',     'CEP68 ≤ 3 m (stationary)',         '≤ 3 m',          f'{cep68:.3f} m',                         g1_pass),
    ('G2','Fix',     'HDOP ≤ 2.0 in ≥ 95% fixes',       '≥ 95%',          f'{g2_pct:.0f}%  mean={hdops.mean():.2f}',g2_pass),
    ('G3','Fix',     'Multi-constellation fix',          '≥ 2 systems',    f'{len(gsa_sys)} systems',                g3_pass),
]

df_sc = pd.DataFrame(scorecard, columns=['ID','Category','Metric','Threshold','Measured','Pass'])
n_pass = df_sc['Pass'].sum()
n_fail = (~df_sc['Pass']).sum()
n_total_m = len(df_sc)

print(f'\n{"="*72}')
print(f'  GNSS QUALITY SCORECARD — Xiaomi 2201116PI | Qualcomm MPSS.HI.4.3.1')
print(f'{"="*72}')
print(f'  Overall: {n_pass}/{n_total_m} metrics PASSED  |  {n_fail}/{n_total_m} FAILED')
print(f'{"="*72}\n')

cat_order = ['Clock','Tracking','Signal','PRR','ADR','Feature','Fix']
for cat in cat_order:
    cat_df = df_sc[df_sc['Category']==cat]
    cp = cat_df['Pass'].sum(); ct2 = len(cat_df)
    print(f'  ── {cat} ({cp}/{ct2}) ───────────────────────────────────────────────')
    for _,row in cat_df.iterrows():
        status = '  PASS  ' if row['Pass'] else '  FAIL  '
        print(f'  {row["ID"]} [{status}] {row["Metric"]:40s} {row["Measured"]}')
    print()

# Summary plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metric-by-metric pass/fail
ax = axes[0]
ids  = df_sc['ID'].tolist()
cols = ['#2ca02c' if p else '#d62728' for p in df_sc['Pass']]
ax.barh(ids, [1]*len(ids), color=cols, alpha=0.85)
for i, (passed, meas) in enumerate(zip(df_sc['Pass'], df_sc['Measured'])):
    ax.text(0.02, i, 'PASS' if passed else 'FAIL', va='center', fontsize=7.5,
            fontweight='bold', color='white')
ax.set_xlim(0, 1.25); ax.set_xticks([])
ax.set_title('Quality Scorecard — All 35 Metrics')
ax.invert_yaxis()

# Category summary donut
ax = axes[1]
cat_pass = df_sc.groupby('Category')['Pass'].agg(['sum','count'])
cat_pass = cat_pass.reindex(cat_order)
cat_labels = [f"{c}\n{int(r['sum'])}/{int(r['count'])}" for c,r in cat_pass.iterrows()]
cat_cols = ['#2ca02c' if r['sum']==r['count'] else
            '#d62728' if r['sum']==0 else '#ff7f0e'
            for _,r in cat_pass.iterrows()]
ax.pie(cat_pass['count'], labels=cat_labels, colors=cat_cols, autopct='',
       startangle=90, wedgeprops=dict(width=0.5))
ax.set_title(f'By Category  ({n_pass}/{n_total_m} PASS)')
patch_pass = mpatches.Patch(color='#2ca02c', label='All PASS')
patch_part = mpatches.Patch(color='#ff7f0e', label='Partial PASS')
patch_fail = mpatches.Patch(color='#d62728', label='All FAIL')
ax.legend(handles=[patch_pass,patch_part,patch_fail], loc='lower right', fontsize=8)

plt.suptitle('GNSS Quality Analysis — Xiaomi 2201116PI (Qualcomm MPSS.HI.4.3.1)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

display(df_sc[['ID','Category','Metric','Threshold','Measured','Pass']]\
        .style.apply(lambda col: ['background-color: #c8efca' if v else
                                  'background-color: #f5c6c6' for v in col],
                     subset=['Pass']))